In [55]:
import tkinter as tk
from tkinter import filedialog, messagebox
import librosa
import numpy as np
import pandas as pd
import joblib
import os 

In [56]:
# Load optimized RandomForest model and RFE selector
model = joblib.load("../models/rf_optimized.pkl")
selector = joblib.load("../models/rfe_selector.pkl")

# All features before feature selection (same as training input)
all_feature_names = [
    'mfcc_1_mean','mfcc_1_std','mfcc_2_mean','mfcc_2_std','mfcc_3_mean','mfcc_3_std',
    'mfcc_4_mean','mfcc_4_std','mfcc_5_mean','mfcc_5_std','mfcc_6_mean','mfcc_6_std',
    'mfcc_7_mean','mfcc_7_std','mfcc_8_mean','mfcc_8_std','mfcc_9_mean','mfcc_9_std',
    'mfcc_10_mean','mfcc_10_std','mfcc_11_mean','mfcc_11_std','mfcc_12_mean','mfcc_12_std',
    'mfcc_13_mean','mfcc_13_std','mfcc_14_mean','mfcc_14_std','mfcc_15_mean','mfcc_15_std',
    'chroma_1_mean','chroma_2_mean','chroma_3_mean','chroma_4_mean','chroma_5_mean',
    'chroma_6_mean','chroma_7_mean','chroma_8_mean','chroma_9_mean','chroma_10_mean',
    'chroma_11_mean','chroma_12_mean','contrast_1_mean','contrast_2_mean','contrast_3_mean',
    'contrast_4_mean','contrast_5_mean','contrast_6_mean','contrast_7_mean',
    'spec_centroid_mean','spec_bandwidth_mean','spec_rolloff_mean',
    'zcr_mean','rms_mean','tempo'
]

In [57]:
# Feature extraction function
def extract_features(filepath):
    y, sr = librosa.load(filepath, sr=22050)
    features = {}

    # MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=15)
    for i in range(mfcc.shape[0]):
        features[f'mfcc_{i+1}_mean'] = np.mean(mfcc[i])
        features[f'mfcc_{i+1}_std'] = np.std(mfcc[i])

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(chroma.shape[0]):
        features[f'chroma_{i+1}_mean'] = np.mean(chroma[i])

    # Spectral contrast
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        features[f'contrast_{i+1}_mean'] = np.mean(contrast[i])

    # Other spectral features
    features['spec_centroid_mean'] = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    features['spec_bandwidth_mean'] = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    features['spec_rolloff_mean'] = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    features['zcr_mean'] = np.mean(librosa.feature.zero_crossing_rate(y))
    features['rms_mean'] = np.mean(librosa.feature.rms(y=y))

    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features['tempo'] = tempo

    return features

# Function to browse and select file
def upload_file():
    global selected_file
    filename = filedialog.askopenfilename(filetypes=[("Audio Files", "*.wav *.mp3")])
    if filename:
        selected_file = filename
        file_label.config(text=f"Selected file: {os.path.basename(filename)}") 

# Function to classify audio
def classify_audio():
    global selected_file
    if not selected_file:
        messagebox.showerror("Error", "Please upload an audio file first!")
        return
    
    try:
        # Extract features
        feats_dict = extract_features(selected_file)
        feats_df = pd.DataFrame([feats_dict], columns=all_feature_names)

        # Apply feature selection
        feats_selected = selector.transform(feats_df)

        # Predict
        prediction = model.predict(feats_selected)[0]

        # Show result
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(feats_selected)[0]
            msg = f"Predicted Genre: {prediction}\n\nProbabilities:\n"
            for label, p in zip(model.classes_, proba):
                msg += f"{label}: {p:.2f}\n"
        else:
            msg = f"Predicted Genre: {prediction}"

        messagebox.showinfo("Result", msg)

    except Exception as e:
        messagebox.showerror("Error", f"Could not process the file:\n{e}")

In [58]:
# GUI
selected_file = None

root = tk.Tk()
root.title("Music Genre Classifier")
root.configure(padx=20, pady=20)
default_font = ('Helvetica', 12)

title_label = tk.Label(root, text="Music Genre Classifier", font=("Helvetica", 16, "bold"))
title_label.pack(pady=10)

file_label = tk.Label(root, text="No file selected", font=default_font, fg="gray")
file_label.pack(pady=10)

upload_button = tk.Button(root, text="Upload File", command=upload_file, font=default_font, width=15)
upload_button.pack(pady=5)

# Classify button 
classify_button = tk.Button(root, text="Classify Audio", command=classify_audio, font=default_font, bg="blue", fg="white", width=15)
classify_button.pack(pady=20)

# Run GUI
root.mainloop()
